# Reddit-Driven Equity Forecast
### End-to-end analysis: data collection → sentiment → ML modelling → evaluation

**Study window:** configurable via `.env` (default 2020-01-01 → 2024-12-31)  
**Target:** predict next-day closing price % move for top-10 tickers  
**Models:** Naive Baseline · XGBoost · LightGBM  
**Metrics:** MAE · RMSE · Directional Accuracy vs Baseline

---
**Acceptance Criteria:**
1. ✅ End-to-end runs without manual tweaks
2. ✅ Dataset coverage proof printed at each stage
3. ✅ Top-10 selection demonstrably by volume

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from config import cfg
print(f'Study window: {cfg.start_date} → {cfg.end_date}')
print(f'Top-N tickers: {cfg.top_n_tickers}')

## Stage 1 — Ticker Selection (Acceptance Criterion 3)

In [ ]:
from src.ticker_selector import TickerSelector
ts = TickerSelector()
volume_df = ts.volume_ranking_df()
top_tickers = ts.get_top_tickers()
print('\nFull volume ranking (proof of criterion 3):')
display(volume_df)
print(f'\nSelected top-{cfg.top_n_tickers} tickers by VOLUME: {top_tickers}')

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
top_df = volume_df.head(cfg.top_n_tickers)
bars = ax.barh(top_df['ticker'][::-1], top_df['avg_daily_volume'][::-1]/1e6, color=sns.color_palette('tab10', len(top_df)))
ax.set_xlabel('Avg Daily Volume (millions)')
ax.set_title(f'Top-{cfg.top_n_tickers} Tickers by 90-Day Average Trading Volume')
for bar, val in zip(bars, top_df['avg_daily_volume'][::-1]/1e6):
    ax.text(val+0.5, bar.get_y()+bar.get_height()/2, f'{val:.0f}M', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Stage 2 — Reddit Data Collection

In [ ]:
raw_path = cfg.data_raw / 'reddit_raw.parquet'
if raw_path.exists():
    reddit_df = pd.read_parquet(raw_path)
    print(f'Loaded cached Reddit data: {len(reddit_df):,} rows')
else:
    from src.reddit_collector import RedditCollector
    rc = RedditCollector()
    reddit_df = rc.run(tickers=top_tickers)
if not reddit_df.empty:
    display(reddit_df.head(3))
    print('\nType distribution:')
    print(reddit_df['type'].value_counts())

## Stage 3 — Sentiment Scoring

In [ ]:
sent_path = cfg.data_processed / 'sentiment_daily.parquet'
if sent_path.exists():
    sentiment_df = pd.read_parquet(sent_path)
    print(f'Loaded cached sentiment: {len(sentiment_df):,} rows')
elif not reddit_df.empty:
    from src.sentiment_engine import SentimentEngine
    se = SentimentEngine(use_finbert=True)
    sentiment_df = se.score_and_aggregate(reddit_df)
else:
    sentiment_df = pd.DataFrame()
    print('No sentiment data — will use market features only.')
if not sentiment_df.empty:
    display(sentiment_df.describe())

## Stage 4 — Market Data (Criterion 2: Coverage Proof)

In [ ]:
mkt_path = cfg.data_processed / 'market_features.parquet'
if mkt_path.exists():
    market_df = pd.read_parquet(mkt_path)
    print(f'Loaded cached market features: {len(market_df):,} rows')
else:
    from src.market_data import MarketDataFetcher
    mdf = MarketDataFetcher()
    market_df = mdf.fetch_and_engineer(top_tickers)
coverage = market_df.groupby('ticker')['date'].agg(min_date='min', max_date='max', trading_days='count')
coverage['years'] = (pd.to_datetime(coverage['max_date']) - pd.to_datetime(coverage['min_date'])).dt.days / 365.25
print('\nMarket Data Coverage Proof:')
display(coverage.round(2))

## Stage 5 — Model Training & Evaluation

Three models:
- **Naive Baseline** — yesterday's return (random-walk benchmark)
- **XGBoost** — gradient-boosted trees  
- **LightGBM** — leaf-wise gradient boosting

Split: 70% train | 10% val (early stopping) | 20% test

In [ ]:
from src.dataset_builder import DatasetBuilder
db = DatasetBuilder()
X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = db.build(
    market_df=market_df,
    sentiment_df=sentiment_df if not sentiment_df.empty else None,
)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Features ({len(feature_cols)}): {feature_cols[:10]} ...')

In [ ]:
from src.models import ModelTrainer
mt = ModelTrainer()
results = mt.train_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, feature_cols)
display(results)

In [ ]:
mt.print_comparison(results)

## Stage 6 — Visualisations

In [ ]:
preds = pd.DataFrame({'actual': y_test})
for name, model in mt.trained_models.items():
    preds[name] = model.predict(X_test)
preds.to_parquet(cfg.outputs_dir / 'test_predictions.parquet', index=False)
ml_models = [c for c in preds.columns if c not in ('actual','Naive Baseline')]
fig, axes = plt.subplots(1, len(ml_models), figsize=(7*len(ml_models), 5))
if len(ml_models)==1: axes=[axes]
for ax,(name,color) in zip(axes, zip(ml_models, sns.color_palette('tab10'))):
    ax.scatter(preds['actual'], preds[name], alpha=0.25, s=7, color=color)
    lim = np.quantile(np.abs(preds['actual']),0.99)*1.1
    ax.plot([-lim,lim],[-lim,lim],'r--',lw=1.2,label='Perfect')
    ax.set_xlim(-lim,lim); ax.set_ylim(-lim,lim)
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted'); ax.set_title(name); ax.legend()
plt.suptitle('Actual vs Predicted Returns (Test Set)')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(14,4))
metrics=[('MAE','Mean Absolute Error'),('RMSE','RMSE'),('DirectionalAccuracy','Directional Accuracy')]
colors=['gray' if m=='Naive Baseline' else '#4C72B0' for m in results['model']]
for ax,(col,label) in zip(axes, metrics):
    vals=results[col].tolist()
    bars=ax.bar(results['model'],vals,color=colors)
    ax.set_title(label)
    for bar,val in zip(bars,vals):
        fmt=f'{val:.1%}' if col=='DirectionalAccuracy' else f'{val:.5f}'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02, fmt, ha='center',va='bottom',fontsize=9)
    if col=='DirectionalAccuracy': ax.axhline(0.5,color='red',ls='--',lw=1)
    ax.set_ylim(0,max(vals)*1.2); ax.tick_params(axis='x',rotation=20)
plt.suptitle('Model Comparison (gray=naive baseline)')
plt.tight_layout(); plt.show()

## Summary

| Stage | Output | Criterion |
|---|---|---|
| Ticker selection | `data/processed/volume_ranking.parquet` | ✅ Criterion 3 |
| Reddit collection | `data/raw/reddit_raw.parquet` | ✅ Criterion 1 |
| Coverage proof | Printed automatically | ✅ Criterion 2 |
| Sentiment | `data/processed/sentiment_daily.parquet` | ✅ Criterion 1 |
| Market features | `data/processed/market_features.parquet` | ✅ Criterion 1 |
| Models | `models/*.pkl` | ✅ Criterion 1 |
| Evaluation | `outputs/model_comparison.csv` | ✅ Criterion 1 |
| Interactive plots | `outputs/*.html` | Bonus |

**Known limitations:** Pushshift API may be unavailable for pre-2023 data. Reddit sentiment = correlation, not causation.